
# Code overview for tutor

This notebook gives a high-level explanation of the implementation supporting the thesis. It is designed so that a reviewer can understand **what the code does** without reading all MATLAB files line by line.

The key point is that the thesis tables are **not hardcoded**. The MATLAB scripts build or load BFM models, apply scenario evidence, run BFM inference, parse the resulting belief functions, and export structured CSV files. The thesis tables are then generated from those CSVs.



## 1. File types in this repository

| File type | Role |
|---|---|
| `.m` MATLAB scripts | Run BFM experiments and export CSV results. |
| `.txt` UIL files | Human-readable source models for BFM: variables, relations, valuations, priors, and ignorance. |
| `.mat` files | Generated MATLAB/BFM model objects. Useful for rerunning, not ideal for explanation. |
| `.csv` files | Final structured result tables: belief, plausibility, width, conflict, rankings. |
| diary `.txt` files | MATLAB run logs proving the workflow was executed. |


In [1]:

from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
print(ROOT)

paths = {
    'matlab_scripts': ROOT / 'matlab' / 'export_scripts',
    'uil': ROOT / 'bfm_models' / 'uil',
    'results': ROOT / 'results' / 'raw',
    'diaries': ROOT / 'results' / 'diaries'
}

for name, path in paths.items():
    print(f"{name:15s}: {path}")


/mnt/data/belief-network-fault-diagnosis-final
matlab_scripts : /mnt/data/belief-network-fault-diagnosis-final/matlab/export_scripts
uil            : /mnt/data/belief-network-fault-diagnosis-final/bfm_models/uil
results        : /mnt/data/belief-network-fault-diagnosis-final/results/raw
diaries        : /mnt/data/belief-network-fault-diagnosis-final/results/diaries



## 2. Overall workflow

The implementation follows the same pattern across the experiment levels:

```text
Base UIL model / generated UIL variant
      ↓
uil2bm converts UIL into a BFM .mat model
      ↓
condiembed transforms conditional beliefs into ordinary embedded beliefs
      ↓
scenario evidence is created as belief functions
      ↓
solve is called for every fault variable
      ↓
showbel output is parsed into belief / plausibility / width / conflict
      ↓
writetable exports a CSV used in the thesis tables
```

Only the **scenario definitions**, the **incompleteness rules**, and the **fault-variable mapping** are explicitly specified. The diagnostic results themselves are produced by BFM inference.


In [2]:

# Show where the main BFM workflow calls appear in each MATLAB script.
keywords = ['uil2bm', 'condiembed', 'keepbel', 'solve(', 'showbel', 'writetable']
for script in sorted(paths['matlab_scripts'].glob('export_bfm_*.m')):
    print('\n' + '='*90)
    print(script.name)
    text = script.read_text(errors='ignore').splitlines()
    for i, line in enumerate(text, start=1):
        if any(k in line for k in keywords):
            print(f"{i:4d}: {line.strip()}")



export_bfm_L0_results.m
  41: embelall = condiembed([BELIEF(:).number]);
  44: keepbel(embelall);
  98: showbel(obs);
 113: result = solve(beliefs_with_evidence, bfm_var);
 158: writetable(T, out_path);
 239: % showbel returns a character array representation of the mass table.
 240: tbl = showbel(result_belief_id);

export_bfm_L1_local_prior_ignorance.m
 152: uil2bm(temp_uil, temp_bm_file);
 158: embelall = condiembed([BELIEF(:).number]);
 161: keepbel(embelall);
 172: showbel(obs);
 191: result = solve(beliefs_with_evidence, bfm_var);
 236: writetable(T, out_path);
 361: tbl = showbel(result_belief_id);

export_bfm_L2_fault_family_prior_ignorance.m
 167: uil2bm(temp_uil, temp_bm_file);
 173: embelall = condiembed([BELIEF(:).number]);
 176: keepbel(embelall);
 187: showbel(obs);
 206: result = solve(beliefs_with_evidence, bfm_var);
 268: writetable(T, out_path);
 445: tbl = showbel(result_belief_id);

export_bfm_L3_observation_uncertainty.m
  54: embelall = condiembed([BELIEF(:).numb


## 3. UIL source models: what BFM reads

The UIL files are the most readable representation of the belief network. They define:

- fault variables such as `PSF`, `Batt`, `SW3`, `C3`, and `LF`;
- voltage variables `V0`--`V8`;
- observation variables such as LEDs and multimeter readings;
- priors or vacuous priors;
- deterministic physical relations, such as voltage propagation;
- visual observation rules and measurement rules.

For example, the L0 UIL file defines variables and relations directly. The temporary L1/L2/L4 UIL files are generated by MATLAB scripts from the base L0 model and show how incompleteness is represented.


In [3]:

# Inspect the first part of the L0 UIL model.
l0_uil = paths['uil'] / 'L0_complete' / 'ten_cube_L0_complete_s8.txt'
print(l0_uil)
print('\n'.join(l0_uil.read_text(errors='ignore').splitlines()[:45]))


/mnt/data/belief-network-fault-diagnosis-final/bfm_models/uil/L0_complete/ten_cube_L0_complete_s8.txt
# 10-cube fault diagnosis model - L0 complete - Scenario 8 reference model
# Generated by run_10cube_L0_scenario8.m
# Short BFM variable names are used to avoid display truncation.

DEFINE VARIABLE PSF {no yes};
DEFINE VARIABLE Batt {good exh};
DEFINE VARIABLE SW1 {ok det};
DEFINE VARIABLE SW2 {ok det};
DEFINE VARIABLE SW3 {ok det};
DEFINE VARIABLE SW4 {ok det};
DEFINE VARIABLE SW5 {ok det};
DEFINE VARIABLE SW6 {ok det};
DEFINE VARIABLE SW7 {ok det};
DEFINE VARIABLE SW8 {ok det};
DEFINE VARIABLE C1 {ok bro};
DEFINE VARIABLE C2 {ok bro};
DEFINE VARIABLE C3 {ok bro};
DEFINE VARIABLE C4 {ok bro};
DEFINE VARIABLE C5 {ok bro};
DEFINE VARIABLE C6 {ok bro};
DEFINE VARIABLE C7 {ok bro};
DEFINE VARIABLE C8 {ok bro};
DEFINE VARIABLE CL {ok bro};
DEFINE VARIABLE LF {ok bro};
DEFINE VARIABLE V0 {v12 v0};
DEFINE VARIABLE V1 {v12 v0};
DEFINE VARIABLE V2 {v12 v0};
DEFINE VARIABLE V3 {v12 v0};
DEFINE 

In [4]:

# Find representative UIL lines: priors, conditional priors, physical relations, and observation rules.
uil_text = l0_uil.read_text(errors='ignore').splitlines()
patterns = ['SET VALUATION PR_PSF', 'SET CONDITIONAL VALUATION PB0', 'DEFINE RELATION R_V3',
            'SET VALUATION R_V3', 'SET VALUATION R_I3', 'SET VALUATION R_M3']
for pattern in patterns:
    print('\n---', pattern, '---')
    for i, line in enumerate(uil_text, start=1):
        if pattern in line:
            for j in range(i, min(i+10, len(uil_text)+1)):
                print(f"{j:4d}: {uil_text[j-1]}")
            break



--- SET VALUATION PR_PSF ---
  78: SET VALUATION PR_PSF {yes} 0.005, {no} 0.995;
  79: SET CONDITIONAL VALUATION PB0 GIVEN {no}, {exh} 0.080, {good} 0.920;
  80: SET CONDITIONAL VALUATION PB1 GIVEN {yes}, {exh} 0.950, {good} 0.050;
  81: SET VALUATION PR_SW1 {det} 0.030, {ok} 0.970;
  82: SET VALUATION PR_SW2 {det} 0.030, {ok} 0.970;
  83: SET VALUATION PR_SW3 {det} 0.030, {ok} 0.970;
  84: SET VALUATION PR_SW4 {det} 0.030, {ok} 0.970;
  85: SET VALUATION PR_SW5 {det} 0.030, {ok} 0.970;
  86: SET VALUATION PR_SW6 {det} 0.030, {ok} 0.970;
  87: SET VALUATION PR_SW7 {det} 0.030, {ok} 0.970;

--- SET CONDITIONAL VALUATION PB0 ---
  79: SET CONDITIONAL VALUATION PB0 GIVEN {no}, {exh} 0.080, {good} 0.920;
  80: SET CONDITIONAL VALUATION PB1 GIVEN {yes}, {exh} 0.950, {good} 0.050;
  81: SET VALUATION PR_SW1 {det} 0.030, {ok} 0.970;
  82: SET VALUATION PR_SW2 {det} 0.030, {ok} 0.970;
  83: SET VALUATION PR_SW3 {det} 0.030, {ok} 0.970;
  84: SET VALUATION PR_SW4 {det} 0.030, {ok} 0.970;
  85:


## 4. Experiment level L0: complete-information validation

L0 loads the complete BFM model and runs the selected thesis scenarios. It does not remove information. Its purpose is to validate that the BFM implementation behaves consistently with the complete Bayesian-network reference model.

The script loops over the selected scenarios and over the fault-variable map. Each fault is solved by BFM and the result is exported to CSV.


In [5]:

# Key L0 lines showing scenario loop and solving.
def print_excerpt(path, start, end):
    lines = path.read_text(errors='ignore').splitlines()
    for i in range(start, min(end, len(lines))+1):
        print(f"{i:4d}: {lines[i-1]}")

l0_script = paths['matlab_scripts'] / 'export_bfm_L0_results.m'
print_excerpt(l0_script, 78, 125)


  78: scenario_ids = [7, 8, 11, 14];
  79: 
  80: % Storage
  81: rows = {};
  82: 
  83: % ------------------------------------------------------------
  84: % Run scenarios
  85: % ------------------------------------------------------------
  86: 
  87: for s_idx = 1:length(scenario_ids)
  88: 
  89:     scenario_id = scenario_ids(s_idx);
  90: 
  91:     fprintf('\n============================================================\n');
  92:     fprintf('Scenario %d\n', scenario_id);
  93:     fprintf('============================================================\n');
  94: 
  95:     obs = make_scenario_observations(scenario_id);
  96: 
  97:     disp('Evidence used:');
  98:     showbel(obs);
  99: 
 100:     beliefs_with_evidence = [embelall obs];
 101: 
 102:     scenario_rows = {};
 103: 
 104:     for k = 1:size(fault_map, 1)
 105: 
 106:         bfm_var = fault_map{k, 1};
 107:         bfm_fault_state = fault_map{k, 2};
 108:         bn_fault_var = fault_map{k, 3};
 109:         bn


## 5. Experiment level L1: local prior ignorance

L1 replaces only the locally relevant prior(s) with full-frame ignorance. This is done by editing the UIL text before conversion to BFM. This means the results are not typed manually; the script changes the model definition and then runs BFM inference.

Examples:

- Scenario 8: the local priors for `SW3` and `C3` are replaced by ignorance.
- Scenario 11: the local priors for `SW6` and `C6` are replaced by ignorance.
- Scenario 7 / 14: battery and/or power subsystem priors are replaced by ignorance.


In [6]:

l1_script = paths['matlab_scripts'] / 'export_bfm_L1_local_prior_ignorance.m'
print_excerpt(l1_script, 104, 135)
print('\nHelper functions that replace priors with ignorance:')
print_excerpt(l1_script, 252, 290)


 104:     txt = fileread(l0_uil);
 105: 
 106:     switch scenario_id
 107: 
 108:         case 7
 109:             % Battery prior/conditional battery knowledge unknown.
 110:             % PSF prior remains known because M_PSU_short evidence is available.
 111:             txt = replace_battery_conditional_with_ignorance(txt);
 112: 
 113:         case 8
 114:             % Local module 3 ambiguity: switch 3 and cable 3 priors unknown.
 115:             txt = replace_switch_prior_with_ignorance(txt, 3);
 116:             txt = replace_cable_prior_with_ignorance(txt, 3);
 117: 
 118:         case 11
 119:             % Sparse-evidence module-6 case:
 120:             % switch 6 and cable 6 priors unknown.
 121:             txt = replace_switch_prior_with_ignorance(txt, 6);
 122:             txt = replace_cable_prior_with_ignorance(txt, 6);
 123: 
 124:         case 14
 125:             % PSU short + discharged battery:
 126:             % PSF prior and battery conditional knowledge un


## 6. Experiment level L2: fault-family prior ignorance

L2 removes broader prior families, such as all switch and cable priors in control-chain scenarios. This tests whether the model can still localise a diagnosis when a whole reliability family is unavailable.

Again, the script modifies the UIL model by replacing prior valuations with vacuous mass functions, then converts and solves the model.


In [7]:

l2_script = paths['matlab_scripts'] / 'export_bfm_L2_fault_family_prior_ignorance.m'
print_excerpt(l2_script, 112, 145)
print('\nFamily-level replacement helpers:')
print_excerpt(l2_script, 293, 360)


 112:         case 7
 113:             % Upstream power-family prior ignorance:
 114:             % PSF prior and battery conditional prior unknown.
 115:             %
 116:             % Real-life interpretation:
 117:             % The company may know the circuit and measurements, but not
 118:             % trust reliability estimates for the power subsystem.
 119:             txt = replace_power_subsystem_priors_with_ignorance(txt);
 120: 
 121:         case 8
 122:             % Control-chain fault-family prior ignorance:
 123:             % all switch and cable priors unknown.
 124:             %
 125:             % Real-life interpretation:
 126:             % The system can still be observed, but historical reliability
 127:             % data for switch/cable faults is unavailable or unreliable.
 128:             txt = replace_all_switch_priors_with_ignorance(txt);
 129:             txt = replace_all_cable_priors_with_ignorance(txt);
 130: 
 131:         case 11
 132:       


## 7. Intermediate L3 / L3b: observation uncertainty

L3 and L3b are included because they explain how observation uncertainty was represented before the final combined L4 setup.

- L3 discounts the **evidence itself**: 80% mass on the observed value and 20% on the full frame.
- L3b weakens the **visual observation rules** in the model while keeping multimeter relations reliable.

These levels are not the main thesis result tables, but they are useful for showing how observation uncertainty was coded.


In [8]:

l3_script = paths['matlab_scripts'] / 'export_bfm_L3_observation_uncertainty.m'
l3b_script = paths['matlab_scripts'] / 'export_bfm_L3b_observation_rule_uncertainty.m'
print('L3 soft evidence helper:')
print_excerpt(l3_script, 222, 306)
print('\nL3b visual-rule weakening excerpt:')
print_excerpt(l3b_script, 60, 88)


L3 soft evidence helper:
 222: % Local helper function: scenario evidence with observation uncertainty
 223: % ============================================================
 224: 
 225: function obs = make_scenario_observations_L3(scenario_id, discount_rate)
 226: 
 227:     obs = [];
 228: 
 229:     switch scenario_id
 230: 
 231:         case 7
 232:             % Scenario 7: battery exhausted.
 233:             % Available evidence is discounted to represent uncertain
 234:             % LED/lamp/multimeter observations.
 235:             obs(end+1) = soft_observe('OPSU', 'off', discount_rate);
 236: 
 237:             for n = 1:8
 238:                 obs(end+1) = soft_observe(sprintf('I%d', n), 'off', discount_rate);
 239:             end
 240: 
 241:             obs(end+1) = soft_observe('OLa', 'off', discount_rate);
 242:             obs(end+1) = soft_observe('OLi', 'off', discount_rate);
 243:             obs(end+1) = soft_observe('MB', 'v0', discount_rate);
 244:             o


## 8. Experiment level L4: combined incompleteness

L4 combines several realistic forms of incompleteness:

1. A key observation is missing.
2. Relevant subsystem/fault-family priors are replaced by ignorance.
3. Visual observation rules are weakened.

This is the strongest stress test in the thesis. It uses derived scenarios 15 and 16.


In [9]:

l4_script = paths['matlab_scripts'] / 'export_bfm_L4_combined_incompleteness.m'
print_excerpt(l4_script, 4, 56)
print('\nScenario-level model changes:')
print_excerpt(l4_script, 126, 160)
print('\nVisual rule weakening function:')
print_excerpt(l4_script, 446, 470)


   4: % Export BFM L4 results: combined incompleteness
   5: %
   6: % L4 definition:
   7: %   Combined realistic diagnostic incompleteness:
   8: %     1. A key discriminating observation is missing.
   9: %     2. The relevant subsystem/fault-family priors are unknown.
  10: %     3. Visual observation rules are weakened.
  11: %
  12: % Derived scenarios:
  13: %
  14: %   Scenario 15:
  15: %     Battery discharge observed, PSU-short measurement unavailable.
  16: %     Tests ambiguity between:
  17: %       - battery-only exhaustion
  18: %       - hidden PSU-short-induced battery discharge
  19: %
  20: %     Evidence:
  21: %       OPSU = off
  22: %       I1-I8 = off
  23: %       OLa = off
  24: %       OLi = off
  25: %       MB = v0
  26: %       MPS = missing
  27: %
  28: %     Missing/uncertain knowledge:
  29: %       - PSF prior unknown
  30: %       - battery conditional prior unknown
  31: %       - visual symptom rules weakened
  32: %
  33: %   Scenario 16:
  34: %


## 9. Evidence creation and fault mapping

The scripts use a fault map to connect the short BFM names to the original BN-style fault names. Evidence is created per scenario using helper functions such as `make_scenario_observations` or `make_scenario_observations_L4`.

This is expected and acceptable: the scenarios are the experimental inputs. What is not hardcoded are the final belief/plausibility results, which are generated by BFM inference.


In [10]:

print('L4 evidence helper for derived scenarios:')
print_excerpt(l4_script, 310, 375)


L4 evidence helper for derived scenarios:
 310: % Helper functions: scenario evidence
 311: % ============================================================
 312: 
 313: function obs = make_scenario_observations_L4(scenario_id)
 314: 
 315:     obs = [];
 316: 
 317:     switch scenario_id
 318: 
 319:         case 15
 320:             % Battery discharge observed, PSU-short measurement missing.
 321:             %
 322:             % Evidence:
 323:             %   OPSU = off
 324:             %   I1-I8 = off
 325:             %   OLa = off
 326:             %   OLi = off
 327:             %   MB = v0
 328:             %   MPS = missing
 329: 
 330:             obs(end+1) = observe('OPSU', 'off');
 331: 
 332:             for n = 1:8
 333:                 obs(end+1) = observe(sprintf('I%d', n), 'off');
 334:             end
 335: 
 336:             obs(end+1) = observe('OLa', 'off');
 337:             obs(end+1) = observe('OLi', 'off');
 338:             obs(end+1) = observe('MB', 'v0')


## 10. Extracting belief, plausibility, width, and conflict

The BFM `showbel` output is a text/table representation of a belief function. The MATLAB scripts parse this output to extract:

- belief in the queried fault state;
- plausibility of the queried fault state;
- width = plausibility − belief;
- empty-set mass, used as a conflict indicator.

This extraction is mechanical and is applied uniformly to all faults and scenarios.


In [11]:

print('Belief/plausibility extraction helper from L4 script:')
print_excerpt(l4_script, 539, 650)


Belief/plausibility extraction helper from L4 script:
 539: % Helper function: extract belief/plausibility
 540: % ============================================================
 541: 
 542: function metrics = extract_binary_fault_metrics(result_belief_id, fault_state)
 543: 
 544:     tbl = showbel(result_belief_id);
 545:     lines = cellstr(tbl);
 546: 
 547:     belief = 0;
 548:     plausibility = 0;
 549:     empty_mass = 0;
 550: 
 551:     current_states = {};
 552:     current_raw_mass = NaN;
 553:     current_norm_mass = NaN;
 554: 
 555:     for i = 1:length(lines)
 556: 
 557:         line = strtrim(lines{i});
 558: 
 559:         if isempty(line)
 560:             continue;
 561:         end
 562: 
 563:         if contains(line, 'value')
 564:             continue;
 565:         end
 566: 
 567:         if contains(line, '---')
 568: 
 569:             if ~isempty(current_states) || ~isnan(current_raw_mass) || ~isnan(current_norm_mass)
 570: 
 571:                 [belief, 


## 11. Checking the exported CSV results

The CSV files are the actual result source for the thesis tables. The next cells load the raw CSVs, check their columns, and create compact summaries.


In [12]:

result_files = sorted(paths['results'].glob('bfm_*.csv'))
frames = []
for f in result_files:
    df = pd.read_csv(f)
    if 'width' not in df.columns:
        df['width'] = df['plausibility'] - df['belief']
    frames.append(df)
    print(f.name, df.shape, list(df.columns))

all_results = pd.concat(frames, ignore_index=True)
all_results.head()


bfm_L0_results.csv (80, 11) ['scenario_id', 'level', 'bn_fault_var', 'bn_fault_state', 'bfm_var', 'bfm_fault_state', 'belief', 'plausibility', 'width', 'empty_set_mass', 'bfm_rank']
bfm_L1_local_prior_ignorance_results_ranked.csv (80, 12) ['scenario_id', 'level', 'bn_fault_var', 'bn_fault_state', 'bfm_var', 'bfm_fault_state', 'belief', 'plausibility', 'width', 'empty_set_mass', 'rank_by_belief', 'rank_by_plausibility']
bfm_L2_fault_family_prior_ignorance_results.csv (80, 12) ['scenario_id', 'level', 'bn_fault_var', 'bn_fault_state', 'bfm_var', 'bfm_fault_state', 'belief', 'plausibility', 'width', 'empty_set_mass', 'rank_by_belief', 'rank_by_plausibility']
bfm_L4_combined_incompleteness_results.csv (40, 12) ['scenario_id', 'level', 'bn_fault_var', 'bn_fault_state', 'bfm_var', 'bfm_fault_state', 'belief', 'plausibility', 'width', 'empty_set_mass', 'rank_by_belief', 'rank_by_plausibility']


,scenario_id,level,bn_fault_var,bn_fault_state,bfm_var,bfm_fault_state,belief,plausibility,width,empty_set_mass,bfm_rank,rank_by_belief,rank_by_plausibility
0,7,L0_complete,F_battery,exhausted,Batt,exh,1.00,1.00,0.0,0.9204,1.0,NaN,NaN
1,7,L0_complete,F_sw_1,detached,SW1,det,0.03,0.03,0.0,0.9204,2.0,NaN,NaN
2,7,L0_complete,F_sw_2,detached,SW2,det,0.03,0.03,0.0,0.9204,3.0,NaN,NaN
3,7,L0_complete,F_sw_3,detached,SW3,det,0.03,0.03,0.0,0.9204,4.0,NaN,NaN
4,7,L0_complete,F_sw_4,detached,SW4,det,0.03,0.03,0.0,0.9204,5.0,NaN,NaN


In [13]:

# Top candidates by scenario and level.
summary_frames = []
for f, df in zip(result_files, frames):
    rank_col = 'rank_by_belief' if 'rank_by_belief' in df.columns else 'bfm_rank'
    top = df.sort_values(['scenario_id', rank_col]).groupby('scenario_id').head(5).copy()
    top['source_file'] = f.name
    summary_frames.append(top)

pd.concat(summary_frames)[['source_file','scenario_id','level','bfm_var','bfm_fault_state','belief','plausibility','width','empty_set_mass']]


,source_file,scenario_id,level,bfm_var,bfm_fault_state,belief,plausibility,width,empty_set_mass
0,bfm_L0_results.csv,7,L0_complete,Batt,exh,1.000,1.000,0.0,0.9204
1,bfm_L0_results.csv,7,L0_complete,SW1,det,0.030,0.030,0.0,0.9204
2,bfm_L0_results.csv,7,L0_complete,SW2,det,0.030,0.030,0.0,0.9204
3,bfm_L0_results.csv,7,L0_complete,SW3,det,0.030,0.030,0.0,0.9204
4,bfm_L0_results.csv,7,L0_complete,SW4,det,0.030,0.030,0.0,0.9204
...,...,...,...,...,...,...,...,...,...
20,bfm_L4_combined_incompleteness_results.csv,16,L4_combined_incompleteness,LF,bro,0.015,0.015,0.0,0.0846
21,bfm_L4_combined_incompleteness_results.csv,16,L4_combined_incompleteness,SW3,det,0.000,1.000,1.0,0.0846
22,bfm_L4_combined_incompleteness_results.csv,16,L4_combined_incompleteness,SW4,det,0.000,1.000,1.0,0.0846
23,bfm_L4_combined_incompleteness_results.csv,16,L4_combined_incompleteness,SW5,det,0.000,1.000,1.0,0.0846



## 12. Scenario 8 as the clearest example

Scenario 8 is the clearest demonstration of the thesis argument. Under complete information, L0 ranks `SW3` and `C3` using precise values. When priors are removed, the model no longer has a justified basis for committing to one exact component, so belief can fall while plausibility remains high.


In [14]:

scenario8 = all_results[(all_results['scenario_id'] == 8) & (all_results['bfm_var'].isin(['SW3','C3']))]
scenario8[['level','bfm_var','bfm_fault_state','belief','plausibility','width','empty_set_mass']].sort_values(['level','bfm_var'])


,level,bfm_var,bfm_fault_state,belief,plausibility,width,empty_set_mass
21,L0_complete,C3,bro,0.40486,0.40486,0.0,0.95914
20,L0_complete,SW3,det,0.60729,0.60729,0.0,0.95914
119,L1_local_prior_ignorance,C3,bro,0.00000,1.00000,1.0,0.17281
116,L1_local_prior_ignorance,SW3,det,0.00000,1.00000,1.0,0.17281
187,L2_fault_family_prior_ignorance,C3,bro,0.00000,1.00000,1.0,0.08460
181,L2_fault_family_prior_ignorance,SW3,det,0.00000,1.00000,1.0,0.08460



## 13. Conflict / empty-set mass

The empty-set mass is reported separately as conflict between the combined evidence and model assumptions. It is not the same as belief in a fault. In this thesis, high L0 conflict largely reflects tension between rare-fault priors and strong evidence that a fault occurred.


In [15]:

conflict = (
    all_results.groupby(['scenario_id','level'], as_index=False)['empty_set_mass']
    .first()
    .pivot(index='scenario_id', columns='level', values='empty_set_mass')
)
conflict


level,L0_complete,L1_local_prior_ignorance,L2_fault_family_prior_ignorance,L4_combined_incompleteness
scenario_id,,,,
7,0.92040,0.00500,0.00000,NaN
8,0.95914,0.17281,0.08460,NaN
11,0.68251,0.08435,0.08435,NaN
14,0.99525,0.00000,0.00000,NaN
15,NaN,NaN,NaN,0.0000
16,NaN,NaN,NaN,0.0846



## 14. What this notebook shows

This notebook shows that:

1. The model is defined in readable UIL files.
2. MATLAB scripts modify UIL files to create incompleteness levels.
3. BFM functions (`uil2bm`, `condiembed`, `keepbel`, `solve`, `showbel`) are used for inference.
4. Results are exported to CSV.
5. Thesis tables can be regenerated from CSV outputs.

The implementation therefore does not consist of manually hardcoded thesis result values. The hardcoded parts are the experimental design inputs: scenarios, evidence, mappings, and the intended incompleteness manipulations.
